# Evaluación Formativa 2: Práctica de Modelamiento Predictivo

**Curso:** MCDI501 — Estadística Computacional para la Toma de Decisiones
**Programa:** Magíster en Ciencia de Datos e Inteligencia Artificial
**Universidad Andrés Bello**

---

## Información del Equipo

| Campo | Información |
|-------|-------------|
| **Número de Grupo** | 8 |
| **Integrante 1** | Matias Manriquez Ortiz |
| **Integrante 2** | Daniel Pérez Ramírez |
| **Integrante 3** | Abigaíl Robles Chávez |
| **Integrante 4** | Roberto Sánchez Saldivia |
| **Docente** | Jean Paul Maidana Gonzalez |
| **Fecha de Entrega** | 25/10/2026 |
| **Dataset Seleccionado** | Predict Students' Dropout and Academic Success (UCI, 2021) — deserción frente a graduación |
| **Repositorio** | https://github.com/MatiasManriquezO/MCDI501-grupo8 |

Base del equipo: `data_con_faltantes.csv` (faltantes MCAR generados en S1). Variable objetivo `Deserta` (1 = *Dropout*, 0 = *Graduate*); los *Enrolled* se excluyen, igual que en S1 y S2.

## Configuración del Entorno

In [1]:
# Librerías básicas
import os
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt

# Modelamiento
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, classification_report,
                             roc_curve, auc)

SEED = 42
np.random.seed(SEED)

# Las figuras van a docs/figuras/F2/ cuando el notebook corre desde notebooks/ del repositorio
FIG = '../docs/figuras/F2' if os.path.isdir('../docs') else '.'
os.makedirs(FIG, exist_ok=True)

# Paleta de S1/S2 (azul = Graduate, naranja = Dropout) para que las figuras sean coherentes con los informes previos
AZUL, NARANJA, GRIS, GRILLA = '#2a78d6', '#eb6834', '#52514e', '#e4e3df'
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'font.size': 9.5, 'axes.titlesize': 10.5, 'axes.titlelocation': 'left',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#8c8b86', 'axes.grid': True, 'axes.axisbelow': True,
    'grid.color': GRILLA, 'grid.linewidth': 0.8,
})

ModuleNotFoundError: No module named 'statsmodels'

---
# PARTE 1: Preparación de Datos para Modelamiento 
## 1.1 Carga y Referencia a S1/S2

### Carga del Dataset

In [ ]:
# La misma base de S1 y S2: la versión con faltantes MCAR (12 % de nota de admisión y de edad), generada una sola vez con semilla 42.
# El notebook busca el archivo en la carpeta actual o en la estructura del repositorio (data/processed/).
def buscar(nombre):
    for ruta in (nombre, f'../data/processed/{nombre}', f'data/processed/{nombre}', f'../data/{nombre}', f'data/{nombre}'):
        if os.path.exists(ruta):
            return ruta
    raise FileNotFoundError(f'No se encontró {nombre}. Copie el archivo junto al notebook o en data/processed/.')

df_completo = pd.read_csv(buscar('data_con_faltantes.csv'))
print(f"Dataset completo: {df_completo.shape[0]:,} filas × {df_completo.shape[1]} columnas")
print(df_completo['Target'].value_counts().to_string())
df_completo.head()

In [ ]:
# Mismas decisiones de S1: Dropout (1) frente a Graduate (0), sin los Enrolled (su desenlace aún no se conoce)
OBJETIVO = 'Deserta'
df = df_completo[df_completo['Target'].isin(['Dropout', 'Graduate'])].copy()
df[OBJETIVO] = (df['Target'] == 'Dropout').astype(int)

# y sin los 152 registros sin ninguna actividad académica en dos semestres (S1 y S2: "mantener exclusión")
cols_actividad = [f'Curricular units {s} sem ({t})' for s in ['1st', '2nd']
                  for t in ['enrolled', 'credited', 'evaluations']]
sin_actividad = (df[cols_actividad] == 0).all(axis=1)
print(f"Registros sin actividad académica excluidos: {sin_actividad.sum()}")
df = df[~sin_actividad].copy()

print(f"Estudiantes analizados: {len(df):,} | desertores: {df[OBJETIVO].sum():,} ({df[OBJETIVO].mean()*100:.1f} %)")

### Resumen de Hallazgos de S1

In [ ]:
# Hallazgos de S1 (informe y cuaderno de la Sumativa 1)
correlaciones_s1 = {
    'Unidades aprobadas 2.º sem.':  {'correlacion_con_target': -0.69},
    'Unidades aprobadas 1.er sem.': {'correlacion_con_target': -0.58},
    'Nota 2.º sem. (sin ceros)':    {'correlacion_con_target': -0.33},
    'Edad al ingresar':             {'correlacion_con_target':  0.29},
    'Tasa de desempleo':            {'correlacion_con_target': -0.01},
}
# Pruebas de hipótesis de S1
pruebas_s1 = {
    'Prueba 1 (aprobadas 2.º sem., Welch)': {'efecto': 'd = -1,94', 'p_valor': '< 0,0001'},
    'Prueba 3 (beca y deserción, χ²)':      {'efecto': 'V de Cramér = 0,32; deserta 13,1 % con beca y 48,0 % sin beca', 'p_valor': '< 0,0001'},
}
# Variables con valores faltantes identificadas en S1 (generados MCAR)
faltantes_s1 = {
    'Admission grade':   {'pct_faltante': 12.56, 'patron': 'MCAR'},
    'Age at enrollment': {'pct_faltante': 11.67, 'patron': 'MCAR'},
}

print("Hallazgos de S1 documentados")
for v, d in correlaciones_s1.items():
    print(f"  {v}: r = {d['correlacion_con_target']}")

**Resumen narrativo de S1:**
- **Variable objetivo:** `Deserta` (Dropout frente a Graduate), con n = 3.478 estudiantes y 38,6 % de desertores. A diferencia de un problema de fraude o morosidad, el desbalance es **moderado**.
- **Variables más asociadas a la deserción:** el desempeño académico del primer año. Unidades aprobadas del 2.º semestre (r = −0,69) y del 1.er semestre (r = −0,58), nota del 2.º semestre sin ceros (r = −0,33) y edad (r = +0,29). Las variables macroeconómicas son prácticamente independientes del desenlace.
- **Variables financieras:** la matrícula al día (94,4 % de deserción entre quienes la tienen atrasada) y la beca (13,1 % con beca frente a 48,0 % sin ella) marcan las mayores diferencias, pero la matrícula puede ser **consecuencia** del abandono, no causa.
- **Problemas identificados:** faltantes MCAR en nota de admisión (12,6 %) y edad (11,7 %); 152 registros sin actividad (excluidos); nota de semestre igual a 0 que en realidad significa "sin nota"; edad muy asimétrica (asimetría 1,97; mediana 20 frente a media 23,6).

### Resumen de Hallazgos de S2

In [ ]:
# Hallazgos de S2: correlaciones validadas mediante bootstrap (IC 95 % que NO incluye 0 => correlación estable)
correlaciones_estables_s2 = [
    ('Unidades aprobadas 2.º sem.', OBJETIVO, [-0.71, -0.66]),
    ('Nota 2.º sem. (sin ceros)',   OBJETIVO, [-0.37, -0.30]),
    ('Edad al ingresar',            OBJETIVO, [ 0.25,  0.32]),
]
# Multicolinealidad real entre las unidades aprobadas de ambos semestres
multicolinealidad_s2 = ('Unidades aprobadas 1.er sem. - 2.º sem.', 0.91, [0.90, 0.92])
# Correlación inestable (el IC incluye el 0)
correlaciones_inestables_s2 = [('Tasa de desempleo', OBJETIVO, [-0.04, 0.03])]
# Pruebas validadas por permutación (10.000 permutaciones)
permutacion_s2 = {'Prueba 1': 0.0001, 'Prueba 3 (beca)': 0.0001, 'MCAR': 0.040}

print("Hallazgos de S2 documentados")
for a, b, ic in correlaciones_estables_s2:
    print(f"  Estable: {a} - {b}, IC bootstrap {ic}")

**Resumen narrativo de S2:**
- **Correlaciones estables:** unidades aprobadas del 2.º semestre (r = −0,69 [−0,71; −0,66]), nota del 2.º semestre (r = −0,33 [−0,37; −0,30]) y edad (r = 0,29 [0,25; 0,32]) tienen intervalos estrechos y lejos de cero.
- **Multicolinealidad real:** las unidades aprobadas de ambos semestres correlacionan r = 0,91 [0,90; 0,92]; S2 recomendó **no usarlas juntas** y priorizar las del 2.º semestre.
- **Correlación inestable:** la tasa de desempleo (IC [−0,04; 0,03] incluye el 0) queda fuera del modelo.
- **Beca:** su asociación con la deserción (Prueba 3) se confirmó por permutación (p = 0,0001) y no depende de supuestos de normalidad. S2 no calculó un IC bootstrap para ella, por lo que se incorpora respaldada por esa prueba.
- **Recomendaciones de S2 para el modelo:** transformar la edad por su asimetría, evaluar el efecto de la matrícula al día (posible fuga de información) y medir la imputación contra los valores originales.

---
## 1.2 Tratamiento Básico de Datos

### Tratamiento de Valores Faltantes

In [ ]:
# Variables del modelo, con nombres cortos para leer mejor las tablas
RENOMBRAR = {
    'Curricular units 2nd sem (approved)': 'aprobadas_2s',
    'Curricular units 2nd sem (grade)':    'nota_2s',
    'Age at enrollment':                   'edad',
    'Scholarship holder':                  'becado',
    'Tuition fees up to date':             'matricula_al_dia',
}
df_modelo = df.rename(columns=RENOMBRAR)
# S1: la nota de semestre igual a 0 coincide con 0 unidades aprobadas; no es una nota real, se trata como "sin nota"
df_modelo['nota_2s'] = df_modelo['nota_2s'].where(df_modelo['nota_2s'] > 0)

# Verificar faltantes actuales en las variables candidatas
candidatas = ['aprobadas_2s', 'nota_2s', 'edad', 'becado', 'matricula_al_dia']
print("Valores faltantes por columna:")
print(df_modelo[candidatas].isnull().sum())
print(f"\nNota 2.º sem. sin nota coincide con 0 aprobadas: "
      f"{(df_modelo['nota_2s'].isna() == (df_modelo['aprobadas_2s'] == 0)).all()}")

**Justificación de la estrategia elegida:**

Seleccionamos **imputación por mediana** porque es simple, robusta a la asimetría de la edad (mediana 20 años frente a media 23,6) y preserva el tamaño completo de la muestra. Como los faltantes de la edad son MCAR (S1 y S2: "los faltantes MCAR reducen la precisión, pero no sesgan"), la mediana no introduce sesgo de selección. La mediana se calcula **solo con el conjunto de entrenamiento** y se aplica luego al de prueba.

Dos advertencias: (1) imputar con una constante reduce artificialmente la variabilidad (se mide más abajo contra los valores originales); (2) los 650 faltantes de la nota del 2.º semestre **no son aleatorios**: corresponden exactamente a estudiantes con 0 unidades aprobadas, y la mediana les asigna una nota que no tuvieron. Es un costo aceptable en la formativa; en S3 se probará un indicador "sin nota".

### División Train/Test

In [ ]:
# Variable objetivo y predictores seleccionados (ver Parte 2.1 para la justificación)
predictores_brutos = ['aprobadas_2s', 'nota_2s', 'edad', 'becado']
X = df_modelo[predictores_brutos]
y = df_modelo[OBJETIVO]

In [ ]:
# Dividir en train (70 %) y test (30 %), estratificado por la variable objetivo (conserva el 38,6 % de desertores)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y)

print(f"Train: {len(X_train):,} observaciones")
print(f"Test:  {len(X_test):,} observaciones")
print("\nDistribución de clases en train:")
print(y_train.value_counts(normalize=True).round(4))
print("\nDistribución de clases en test:")
print(y_test.value_counts(normalize=True).round(4))

### Imputación y transformación (ajustadas solo con train)

In [ ]:
# Medianas calculadas SOLO en entrenamiento
medianas = X_train[['nota_2s', 'edad']].median()
print("Medianas de entrenamiento usadas para imputar:")
print(medianas.round(2))

X_train = X_train.fillna(medianas).copy()
X_test = X_test.fillna(medianas).copy()

# Transformar la edad con logaritmo (recomendación S1/S2 por su asimetría y cola)
for d in (X_train, X_test):
    d['log_edad'] = np.log(d['edad'])

predictores = ['aprobadas_2s', 'nota_2s', 'log_edad', 'becado']
X_train = X_train[predictores]
X_test = X_test[predictores]
print(f"\nFaltantes restantes en train: {X_train.isnull().sum().sum()} | en test: {X_test.isnull().sum().sum()}")

In [ ]:
# ¿Cuánto error introduce la mediana? Se compara con los valores originales guardados en S1 (valores_originales.csv)
originales = pd.read_csv(buscar('valores_originales.csv'))
orig_edad = (originales[(originales['variable'] == 'Age at enrollment')
                        & originales['id_estudiante'].isin(df['id_estudiante'])]
             .set_index('id_estudiante')['valor_original'])
mediana_edad = medianas['edad']          # la misma mediana de entrenamiento usada para imputar
edad_obs = df_modelo['edad']
edad_imp = edad_obs.fillna(mediana_edad)

print(f"Edades imputadas evaluadas contra su valor original: {len(orig_edad)}")
print(f"Mediana usada: {mediana_edad:.1f} años")
print(f"Error absoluto medio de la imputación: {np.abs(orig_edad - mediana_edad).mean():.2f} años")
print(f"Desviación estándar de la edad: {edad_obs.std():.2f} (observada) -> {edad_imp.std():.2f} (tras imputar)")
print(f"Media de la edad: {edad_obs.mean():.2f} (observada) frente a {orig_edad.mean():.2f} (valores originales perdidos)")

### Estandarización

In [ ]:
# Estandarizar solo las variables continuas, para que sus odds ratios sean comparables (por desviación estándar).
# La variable binaria 'becado' se deja en 0/1: su odds ratio compara becados con no becados.
continuas = ['aprobadas_2s', 'nota_2s', 'log_edad']
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[continuas] = scaler.fit_transform(X_train[continuas])
X_test_scaled[continuas] = scaler.transform(X_test[continuas])

print("Variables continuas estandarizadas (media 0, desviación 1); 'becado' se mantiene en 0/1")
print("Desviación estándar original de cada variable continua (para interpretar los odds ratios):")
print(pd.Series(scaler.scale_, index=continuas).round(3))
X_train_scaled.describe().round(2).loc[['mean', 'std']]

---
# PARTE 2: Modelo de Regresión Logística 
## 2.1 Selección de Variables 

### Justificación de Variables Seleccionadas

Basándonos en S1 y S2, seleccionamos **4 variables predictoras**:

| Variable | Correlación con Target (S1) | Validación en S2 | Justificación |
|----------|-----------------------------|------------------|---------------|
| Unidades aprobadas, 2.º sem. | −0,69 | IC bootstrap [−0,71; −0,66], estable | Asociación más fuerte y más estable; señal temprana para priorizar tutorías |
| Nota 2.º sem. (sin ceros) | −0,33 | IC bootstrap [−0,37; −0,30], estable | Aporta la calidad del desempeño, además de la cantidad de unidades aprobadas |
| Edad (logaritmo) | +0,29 | IC bootstrap [0,25; 0,32], estable | Los mayores desertan más; se transforma por su asimetría |
| Beca | V de Cramér = 0,32 (Prueba 3) | Permutación p = 0,0001 (Prueba 3) | Variable de apoyo financiero; deserta 13,1 % con beca y 48,0 % sin ella |

**Se descartan:**
- **Unidades aprobadas del 1.er semestre:** r = 0,91 con las del 2.º semestre (S2: multicolinealidad real). Se prioriza el 2.º semestre.
- **Tasa de desempleo:** correlación inestable (IC [−0,04; 0,03] incluye 0) y asociación nula.
- **Matrícula al día:** puede ser consecuencia del abandono (quien deserta deja de pagar). Se evalúa aparte, como sensibilidad, más abajo.
- **Nota de admisión:** no fue validada por S2 como correlación estable con la deserción; queda para la comparación de modelos de S3.

In [ ]:
# Variables seleccionadas (mismas que en la división train/test)
variables_modelo = predictores
print("Variables seleccionadas para el modelo:")
for var in variables_modelo:
    print(f"  - {var}")

---
## 2.2 Ajuste del Modelo

In [ ]:
# Ajustar modelo de regresión logística con statsmodels
X_train_sm = sm.add_constant(X_train_scaled)
modelo = sm.Logit(y_train, X_train_sm).fit(disp=0)
print(modelo.summary())

### Interpretación de Coeficientes

In [ ]:
# Calcular Odds Ratios con su intervalo de confianza del 95 %
ic = np.exp(modelo.conf_int())
interpretacion = pd.DataFrame({
    'Coeficiente': modelo.params,
    'Odds Ratio': np.exp(modelo.params),
    'IC 95 % inf.': ic[0],
    'IC 95 % sup.': ic[1],
    'p-valor': modelo.pvalues,
    'Significativo (α=0.05)': modelo.pvalues < 0.05
}).round(4)

print("Interpretación del modelo:")
interpretacion

In [ ]:
# Equivalencia de los odds ratios en unidades originales
dp = pd.Series(scaler.scale_, index=continuas)
or_por_unidad = np.exp(modelo.params['aprobadas_2s'] / dp['aprobadas_2s'])
print(f"Unidades aprobadas: 1 DE = {dp['aprobadas_2s']:.2f} unidades; OR por unidad adicional = {or_por_unidad:.3f}")
print(f"Nota 2.º sem.:      1 DE = {dp['nota_2s']:.2f} puntos")
print(f"Edad (log):         1 DE = {dp['log_edad']:.3f} en logaritmo (≈ multiplicar la edad por {np.exp(dp['log_edad']):.2f})")
print(f"Pseudo R² de McFadden: {modelo.prsquared:.3f}")

### Interpretación Narrativa

- **Unidades aprobadas, 2.º sem. (OR ≈ 0,09 por desviación estándar):** cada desviación estándar adicional (≈ 3,1 unidades) **reduce las odds de deserción en ≈ 91 %**; por unidad adicional el OR es ≈ 0,45. Es el factor dominante, coherente con S1 (r = −0,69) y S2 (IC estrecho).
- **Nota 2.º sem. (OR ≈ 0,69):** a igualdad de unidades aprobadas, una desviación estándar más de nota (≈ 1,26 puntos) reduce las odds en ≈ 31 %.
- **Edad, en logaritmo (OR ≈ 1,40):** la edad **aumenta** las odds de deserción. Una desviación estándar del logaritmo equivale a multiplicar la edad por ≈ 1,30 (por ejemplo, de 20 a 26 años) y sube las odds en ≈ 40 %.
- **Beca (OR ≈ 0,33):** a igualdad del resto, los becados tienen ≈ 67 % menos odds de desertar que los no becados. Es una asociación observacional: no prueba que la beca evite la deserción.

Las cuatro variables resultan significativas (p < 0,001) y ningún IC 95 % de los odds ratios incluye el 1. El 95 % es una propiedad del procedimiento: si se repitiera el muestreo, el 95 % de los intervalos construidos así contendría el odds ratio verdadero. El pseudo R² de McFadden (≈ 0,52) indica un ajuste muy bueno para un modelo de cuatro variables.

---
## 2.3 Diagnóstico Básico

In [ ]:
# Calcular VIF para evaluar multicolinealidad
X_vif = sm.add_constant(X_train_scaled)
vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).round(2)

print("Factor de Inflación de Varianza (VIF):")
print(vif_data.to_string(index=False))
print("\nInterpretación: VIF > 5 indica posible multicolinealidad problemática (se ignora la constante)")

In [ ]:
# Por qué se excluyó el 1.er semestre: VIF aproximado si se incluyeran ambas variables de unidades aprobadas
r = df_modelo['aprobadas_2s'].corr(df['Curricular units 1st sem (approved)'])
print(f"Correlación entre unidades aprobadas de ambos semestres: r = {r:.2f}")
print(f"VIF teórico con solo esas dos variables: 1 / (1 - r²) = {1/(1-r**2):.1f}")

### Análisis de Multicolinealidad

**¿Hay problemas de multicolinealidad?**

No. Los cuatro VIF quedan entre 1,06 y 1,12, muy por debajo de 5. Eso era esperable: la advertencia de S1/S2 (r = 0,91 entre unidades aprobadas de ambos semestres, VIF teórico ≈ 5,7 solo con esas dos) se resolvió **antes** de ajustar, al dejar únicamente las del 2.º semestre.

**Si hay problemas, ¿qué harías en S3?**

En S3 revisaremos el VIF sobre un modelo más amplio (por ejemplo, con la nota de admisión, las variables financieras y las demográficas). Si reaparece colinealidad, se conservará la variable más estable según S2 o se combinarán variables. También se ampliará el diagnóstico con la linealidad en el logit y las observaciones influyentes (en particular las edades de 40 años o más).

### Sensibilidad: ¿qué pasa si se incorpora la matrícula al día?

S1 advirtió que la matrícula al día puede ser consecuencia del abandono (**fuga de información**). Se compara el modelo con y sin ella usando la misma partición.

In [ ]:
def ajustar(columnas_extra):
    # Replica el flujo completo con columnas extra: imputación con train, log de la edad, estandarización y ajuste
    cols = ['aprobadas_2s', 'nota_2s', 'edad', 'becado'] + columnas_extra
    tr, te = df_modelo.loc[X_train.index, cols].copy(), df_modelo.loc[X_test.index, cols].copy()
    med = tr[['nota_2s', 'edad']].median()
    tr, te = tr.fillna(med), te.fillna(med)
    for d in (tr, te):
        d['log_edad'] = np.log(d['edad'])
    cols = ['aprobadas_2s', 'nota_2s', 'log_edad', 'becado'] + columnas_extra
    tr, te = tr[cols].copy(), te[cols].copy()
    sc = StandardScaler().fit(tr[continuas])
    tr[continuas], te[continuas] = sc.transform(tr[continuas]), sc.transform(te[continuas])
    m = sm.Logit(y_train, sm.add_constant(tr)).fit(disp=0)
    p = m.predict(sm.add_constant(te, has_constant='add'))
    return m, p

m_base, p_base = ajustar([])
m_mat, p_mat = ajustar(['matricula_al_dia'])
comparacion = pd.DataFrame({
    'AUC (test)': [auc(*roc_curve(y_test, p_base)[:2]), auc(*roc_curve(y_test, p_mat)[:2])],
    'Recall (umbral 0,5)': [recall_score(y_test, p_base >= 0.5), recall_score(y_test, p_mat >= 0.5)],
}, index=['Modelo sin matrícula al día', 'Modelo con matrícula al día']).round(3)
print(comparacion)
print(f"\nOdds ratio de la matrícula al día: {np.exp(m_mat.params['matricula_al_dia']):.3f}")

**Lectura:** incorporar la matrícula al día mejora muy poco el AUC (≈ +0,008) y casi no cambia el recall, pero su efecto es muy fuerte (OR ≈ 0,04: quien tiene la matrícula al día tiene odds de desertar 96 % menores; la variable queda en 0/1, sin estandarizar) precisamente porque quien abandona deja de pagar. Como el modelo busca **anticipar** la deserción, se mantiene **fuera** del modelo principal para evitar fuga de información.

---
# PARTE 3: Evaluación de Desempeño
## 3.1 Predicciones y Matriz de Confusión

In [ ]:
# Generar predicciones en test
X_test_sm = sm.add_constant(X_test_scaled, has_constant='add')
y_prob = modelo.predict(X_test_sm)

# Clases predichas (umbral 0.5)
y_pred = (y_prob >= 0.5).astype(int)
print(f"Predicciones generadas para {len(y_pred):,} observaciones")

In [ ]:
# Construir y visualizar matriz de confusión
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5.6, 4.6))
vmax = cm.max()
ax.imshow(cm, cmap='Blues', aspect='auto', vmin=0, vmax=vmax)
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, f'{v:,}', ha='center', va='center',
            color='white' if v / vmax > 0.5 else GRIS, fontsize=13, fontweight='bold')
ax.set_xticks([0, 1]); ax.set_xticklabels(['Graduate', 'Dropout'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['Graduate', 'Dropout'])
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title('Matriz de confusión (test, umbral 0,5)')
ax.grid(False)
plt.tight_layout()
plt.savefig(f'{FIG}/f2_fig_confusion.png')
plt.show()

print(f"\nVerdaderos Negativos (TN): {cm[0,0]:,}")
print(f"Falsos Positivos   (FP): {cm[0,1]:,}")
print(f"Falsos Negativos   (FN): {cm[1,0]:,}")
print(f"Verdaderos Positivos(TP): {cm[1,1]:,}")

---
## 3.2 Métricas de Desempeño

In [ ]:
# Calcular métricas de desempeño (clase positiva = Dropout)
metricas = {
    'Accuracy':  accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall':    recall_score(y_test, y_pred),
    'F1-Score':  f1_score(y_test, y_pred)
}

print("=" * 40)
print("MÉTRICAS DE DESEMPEÑO (test)")
print("=" * 40)
for metrica, valor in metricas.items():
    print(f"{metrica:12s}: {valor:.4f}")

# Referencia: accuracy de un clasificador trivial que predice siempre "Graduate"
print(f"\nAccuracy del clasificador trivial (siempre Graduate): {1 - y_test.mean():.4f}")

In [ ]:
# Reporte de clasificación completo
print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['Graduate', 'Dropout']))

### Interpretación de Métricas

**Accuracy (≈ 89,5 %):** el modelo clasifica bien casi nueve de cada diez estudiantes. Aquí la accuracy **sí es informativa**, porque el desbalance es moderado (38,6 % de desertores) y el clasificador trivial que dice "nadie deserta" alcanzaría solo ≈ 61,4 %. Aun así, no basta por sí sola: no distingue el tipo de error.

**Precision (clase Dropout, ≈ 90,4 %):** de los 363 estudiantes que el modelo marca en riesgo, 328 desertan realmente y 35 no. Hay pocos falsos positivos: apoyar a quien no lo necesitaba es un costo bajo.

**Recall (clase Dropout, ≈ 81,4 %):** el modelo detecta 328 de los 403 desertores reales; **75 pasan sin alerta**. En este problema el recall importa más que la precisión: S1 plantea que no detectar a un estudiante en riesgo cuesta más que apoyar a quien no lo necesitaba (falso negativo más costoso que falso positivo).

**F1-Score (≈ 0,856):** resume el balance entre precision y recall de la clase Dropout.

---
## 3.3 Curva ROC y AUC 

In [ ]:
# Generar curva ROC y calcular AUC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.fill_between(fpr, tpr, color=AZUL, alpha=0.12, zorder=1)
ax.plot(fpr, tpr, color=AZUL, lw=2.2, label=f'Curva ROC (AUC = {roc_auc:.3f})', zorder=3)
ax.plot([0, 1], [0, 1], color=NARANJA, lw=1.3, linestyle='--', label='Clasificador aleatorio', zorder=2)
ax.set_xlabel('Tasa de Falsos Positivos (1 − especificidad)')
ax.set_ylabel('Tasa de Verdaderos Positivos (sensibilidad)')
ax.set_title('Curva ROC (test)')
ax.legend(loc='lower right', frameon=False)
plt.tight_layout()
plt.savefig(f'{FIG}/f2_fig_roc.png')
plt.show()

print(f"AUC = {roc_auc:.4f}")

### Interpretación del AUC

**Valor de AUC obtenido:** ≈ 0,925.

**Interpretación según la escala:**
- AUC = 0.5: No mejor que azar
- AUC = 0.6–0.7: Pobre
- AUC = 0.7–0.8: Aceptable
- AUC = 0.8–0.9: Bueno
- **AUC > 0.9: Excelente** ← nuestro caso

**¿Qué significa en el contexto del problema?** Si tomamos al azar un estudiante que desertó y otro que se graduó, el modelo asigna mayor probabilidad de deserción al que desertó ≈ 92,5 % de las veces. Es una capacidad discriminante excelente, lograda con solo cuatro variables y un modelo lineal.

**Cautela:** gran parte del poder predictivo viene de las unidades aprobadas del 2.º semestre, y un estudiante que abandona a mitad de semestre termina con pocas o ninguna unidad aprobada. Parte de la señal puede ser, entonces, **consecuencia** del abandono y no solo un aviso previo. Por eso el modelo sirve para priorizar tutorías al cierre del primer año, pero no para detectar la deserción antes de que ocurra.

---
# PARTE 4: Síntesis y Reflexión

Responde las siguientes preguntas de reflexión:

## 4.1 Conexión con S1

**¿Cómo usaste los hallazgos de S1 para seleccionar variables?**

Partimos de la matriz de correlaciones de S1: el desempeño académico del primer año es lo más asociado a la deserción (unidades aprobadas del 2.º semestre r = −0,69, nota r = −0,33) y la edad tiene asociación moderada (r = 0,29). La beca entró por la Prueba 3 de S1 (deserta 13,1 % con beca y 48,0 % sin ella). Descartamos la tasa de desempleo, que S1 mostró prácticamente independiente del desenlace. También reutilizamos las decisiones de datos de S1: Dropout frente a Graduate, exclusión de los 152 registros sin actividad, nota 0 tratada como "sin nota" y edad transformada por su asimetría.

---

## 4.2 Conexión con S2

**¿Cómo usaste la validación de S2 para informar tu modelo?**

S2 confirmó, mediante IC bootstrap, que las correlaciones de unidades aprobadas, nota y edad eran **estables** (intervalos sin cruzar 0), y que la tasa de desempleo era inestable (su IC incluye 0). Además, S2 mostró que las unidades aprobadas de ambos semestres tienen r = 0,91: usamos solo las del 2.º semestre, y el VIF final (≤ 1,12) lo confirma. La permutación de S2 respaldó la Prueba 3, base de la inclusión de la beca. Por último, S2 recomendó evaluar la matrícula al día por fuga de información; lo hicimos como sensibilidad y la dejamos fuera.

---

## 4.3 Evaluación del Desempeño

**¿El modelo tiene buen desempeño? ¿Por qué sí o por qué no?**

Sí, con matices. El AUC (≈ 0,925) es excelente y la accuracy (≈ 89,5 %) supera con holgura al clasificador trivial (≈ 61,4 %). El recall de los desertores (≈ 81,4 %) deja sin alerta a 75 de 403 estudiantes, lo que en una decisión de tutorías es el error más costoso. Además, parte de la señal puede ser consecuencia del abandono (unidades aprobadas del 2.º semestre), por lo que el modelo prioriza apoyos al cierre del primer año, no anticipa más temprano.

---

## 4.4 Mejoras para S3

**¿Qué mejorarías en la Sumativa 3?**

Marca las que apliquen y explica:
- [x] **Probar otras variables**: incorporar la nota de admisión, las variables financieras y demográficas, y comparar con este modelo reducido.
- [x] **Usar otra estrategia de imputación**: reemplazar la mediana por **imputación por regresión**, medir el error contra los valores originales y probar un indicador "sin nota" para el 2.º semestre.
- [x] **Ajustar múltiples modelos**: construir 3 modelos (basado en S1/S2, stepwise y AIC/BIC) y elegir el mejor por métricas.
- [x] **Aplicar bootstrap a los coeficientes**: construir IC bootstrap del 95 % para coeficientes y odds ratios, y verificar su estabilidad.
- [x] **Mejorar el recall**: ajustar el umbral de decisión (por debajo de 0,5) para reducir los desertores sin alerta.

---

## 4.5 Resumen Final

**En 3-5 oraciones, resume tu experiencia con esta formativa:**

Esta formativa nos permitió recorrer el flujo completo de modelamiento predictivo con la base de deserción del grupo 8: preparar los datos apoyándonos en S1 y S2, ajustar un modelo logístico de cuatro variables, interpretar odds ratios, diagnosticar multicolinealidad y evaluar el desempeño con matriz de confusión, métricas y curva ROC. El modelo logra un AUC de ≈ 0,925 y un recall de ≈ 81 % para los desertores. El principal aprendizaje fue que una métrica alta no basta: hay que revisar el tipo de error (75 desertores sin alerta) y si el predictor principal es causa o consecuencia del abandono. Identificamos las mejoras para S3: tres modelos, imputación por regresión, bootstrap de coeficientes y ajuste del umbral de decisión.

---